# Strategist Task — Portfolio Insurance & the 1987 Crash

**Week 0 Submission**

**Memorandum**

| | |
|---|---|
| **To:** | Mr. Sam |
| **From:** | Duy Anh |
| **Date:** | March 17, 2026 |
| **Subject:** | Why Portfolio Insurance Algorithms Sold Into the 1987 Crash |

---

**Contents**
1. Setup & Data Loading
2. Minute-by-Minute Price Chart (Figure 1 — Memo Chart)
3. Analytical Detail Chart (Figure 2 — Appendix)
4. Key Statistics
5. Memo — Full Text
6. References


## 1. Setup & Data Loading

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
import pathlib
NB_DIR   = pathlib.Path(r"d:\Quant Finance\Quant Program\Week 0\Strategist Task")
DATA_PATH    = NB_DIR / "data" / "1987_crash_market_data.csv"
MEMO_PNG     = NB_DIR / "outputs" / "memo_chart.png"
APPENDIX_PNG = NB_DIR / "outputs" / "appendix_chart.png"
OUTPUTS_DIR  = NB_DIR / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print(f"Data file : {DATA_PATH}  exists={DATA_PATH.exists()}")
print(f"Memo chart: {MEMO_PNG}   exists={MEMO_PNG.exists()}")


In [ ]:
# ── Load & clean ──────────────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH)
df_raw["Timestamp"] = pd.to_datetime(df_raw["Timestamp"], format="%m/%d/%Y %H:%M")
df_raw = df_raw.set_index("Timestamp").sort_index()

for col in ["DOW_Futures", "SP500_Futures"]:
    df_raw[col] = pd.to_numeric(df_raw[col].astype(str).str.replace(",", ""), errors="coerce")

# Trading hours only (Mon–Fri 09:30–16:00)
mask = (
    (df_raw.index.dayofweek < 5) &
    (df_raw.index.time >= pd.Timestamp("09:30").time()) &
    (df_raw.index.time <= pd.Timestamp("16:00").time())
)
df = df_raw[mask].copy()

print(f"Rows loaded (trading hours): {len(df):,}")
print(f"Sessions: {df.index.normalize().nunique()} trading days")
for day in sorted(df.index.normalize().unique()):
    sess = df.loc[day.strftime("%Y-%m-%d")]
    sp = sess["SP500_Futures"].dropna()
    print(f"  {day.strftime('%a %b %d')}: {len(sess)} rows | "
          f"SP500 {sp.min():.1f} – {sp.max():.1f} | NaN: {sess['SP500_Futures'].isna().sum()}")


## 2. Minute-by-Minute S&P 500 Price Chart

The chart below plots every minute of live trading across the four sessions of the 1987 crash week.
The market "broke" at **09:42 on October 19**, identified by the convergence of two signals:
the first minute exceeding 3× the pre-crash volatility baseline, and the end of the worst
60-minute loss window. Red shading marks the crisis session; yellow bands mark trading halts.


In [ ]:
# ── Analytics ────────────────────────────────────────────────────────────
price    = df["SP500_Futures"]
returns  = price.pct_change(fill_method=None)

# Clean overnight boundary returns
returns_clean = returns.copy()
for tday in sorted(df.index.normalize().unique()):
    sess = df.loc[tday.strftime("%Y-%m-%d")]
    returns_clean.loc[sess.index[0]] = np.nan

rolling_peak = price.cummax()
drawdown     = (price - rolling_peak) / rolling_peak
rolling_vol  = returns_clean.rolling(30).std()

# Pre-crash baseline vol (Oct 16 trading hours only)
pre_mask = (
    (rolling_vol.index.date < pd.Timestamp("1987-10-19").date()) &
    (rolling_vol.index.time >= pd.Timestamp("09:30").time()) &
    (rolling_vol.index.time <= pd.Timestamp("16:00").time())
)
baseline_vol = rolling_vol[pre_mask].dropna()
baseline_vol = baseline_vol[baseline_vol > 0].median()
threshold    = 3 * baseline_vol

oct19_vol  = rolling_vol[rolling_vol.index.date >= pd.Timestamp("1987-10-19").date()]
triggered  = oct19_vol[oct19_vol > threshold]
break_by_vol = triggered.index[0] if len(triggered) else None

rolling_60   = returns.rolling(60).sum()
break_by_loss = rolling_60.idxmin()

candidates   = [t for t in [break_by_loss, break_by_vol] if t is not None]
break_point  = min(candidates)
print(f"Market break point : {break_point}")
print(f"Max drawdown       : {drawdown.min():.2%}  at {drawdown.idxmin()}")
print(f"Peak vol ratio     : {rolling_vol.max()/baseline_vol:.1f}× pre-crash baseline")
print(f"Worst 60-min move  : {rolling_60.min():.2%}")


In [ ]:
# ── Build integer x-axis helpers ─────────────────────────────────────────
x = np.arange(len(df))
trading_days = sorted(df.index.normalize().unique())

def dt_to_x(ts):
    if ts in df.index:
        return df.index.get_loc(ts)
    pos = df.index.searchsorted(ts)
    return min(pos, len(df) - 1)

sessions = []
for tday in trading_days:
    rows = df[df.index.normalize() == tday]
    sp   = rows["SP500_Futures"].dropna()
    sessions.append({
        "date"   : tday,
        "label"  : tday.strftime("%a\n%b %d"),
        "x_start": dt_to_x(rows.index[0]),
        "x_end"  : dt_to_x(rows.index[-1]),
        "x_mid"  : (dt_to_x(rows.index[0]) + dt_to_x(rows.index[-1])) // 2,
    })

oct19_sess = [s for s in sessions if s["date"].date() == pd.Timestamp("1987-10-19").date()]
bp_x       = dt_to_x(break_point)
bp_price   = price.iloc[bp_x]
max_dd_x   = dt_to_x(drawdown.idxmin())

halt_periods = [
    (pd.Timestamp("1987-10-19 12:00"), pd.Timestamp("1987-10-19 13:00")),
    (pd.Timestamp("1987-10-20 11:30"), pd.Timestamp("1987-10-20 13:00")),
]
print("Helpers built.")


In [ ]:
# ── Figure 1: Memo chart (light theme, single panel) ──────────────────────
MEMO_L_BG   = "#ffffff"; MEMO_P_BG = "#f8f9fa"; MEMO_ALT_BG = "#f0f2f5"
MEMO_GRID   = "#dee2e6"; MEMO_TEXT = "#212529"; MEMO_DIM = "#6c757d"
MEMO_BLUE   = "#1a56db"; MEMO_RED_BG = "#fee2e2"; MEMO_RED_ANN = "#dc2626"
MEMO_SEP    = "#adb5bd"; MEMO_HALT_BG = "#fff3cd"

fig_m, ax_m = plt.subplots(1, 1, figsize=(12, 4))
fig_m.patch.set_facecolor(MEMO_L_BG)
ax_m.set_facecolor(MEMO_P_BG)
ax_m.tick_params(colors=MEMO_TEXT, labelsize=9)
ax_m.grid(True, color=MEMO_GRID, linewidth=0.4, linestyle="--", alpha=0.7)
for spine in ax_m.spines.values():
    spine.set_edgecolor(MEMO_GRID)

for i, sess in enumerate(sessions):
    bg = MEMO_ALT_BG if i % 2 == 1 else MEMO_P_BG
    ax_m.axvspan(sess["x_start"], sess["x_end"] + 1, color=bg, alpha=1.0, zorder=0)

if oct19_sess:
    s19 = oct19_sess[0]
    ax_m.axvspan(s19["x_start"], s19["x_end"] + 1, color=MEMO_RED_BG, alpha=1.0, zorder=1)

for halt_start, halt_end in halt_periods:
    if halt_start in df.index or halt_end in df.index:
        ax_m.axvspan(dt_to_x(halt_start), dt_to_x(halt_end) + 1,
                     color=MEMO_HALT_BG, alpha=0.9, linewidth=1.2,
                     edgecolor="#ff9800", linestyle="-", zorder=2)

for sess in sessions[1:]:
    ax_m.axvline(sess["x_start"] - 0.5, color=MEMO_SEP, linewidth=0.8, linestyle="-", zorder=5)

ax_m.plot(x, price.values, color=MEMO_BLUE, linewidth=1.3, zorder=3)

ax_m.annotate(
    f"Market break\n{break_point.strftime('%b %d  %H:%M')}",
    xy=(bp_x, bp_price), xytext=(bp_x + 45, bp_price + 8),
    arrowprops=dict(arrowstyle="->", color=MEMO_RED_ANN, lw=1.5),
    color=MEMO_RED_ANN, fontsize=9, fontweight="bold", zorder=7
)

halt_legend = Patch(facecolor=MEMO_HALT_BG, edgecolor="#ff9800", label="Trading halts")
ax_m.legend(handles=[halt_legend], loc="lower left", fontsize=8, framealpha=0.95)

ax_m.set_xticks([s["x_mid"] for s in sessions])
ax_m.set_xticklabels([s["label"] for s in sessions], color=MEMO_TEXT, fontsize=9)
ax_m.set_xlim(-5, len(df) + 5)
ax_m.set_ylabel("S&P 500 Index Level", color=MEMO_TEXT, fontsize=10)
ax_m.set_title("S&P 500 Futures — Black Monday, October 1987",
               color=MEMO_TEXT, fontsize=12, fontweight="bold", pad=10)
fig_m.text(0.5, -0.02,
           "Source: 1987_crash_market_data.csv  |  S&P 500 Futures, trading hours only (09:30–16:00)",
           ha="center", color=MEMO_DIM, fontsize=7)

plt.tight_layout()
plt.savefig(MEMO_PNG, dpi=150, bbox_inches="tight", facecolor=MEMO_L_BG)
plt.show()
print(f"Figure 1 saved → {MEMO_PNG}")


## 3. Analytical Detail Chart (Appendix — Figure 2)

Three-panel dark-theme chart showing:
- **Panel 1 — 1-min returns** (capped ±1.5%): the crash shows as sustained red fills on Oct 19–20
- **Panel 2 — Cumulative drawdown**: peak-to-trough of **28.6%** reached Oct 20 at 13:01
- **Panel 3 — 30-min rolling volatility**: spikes to **8.3× pre-crash baseline** — the quantitative signature of market dysfunction


In [ ]:
# ── Figure 2: Appendix chart (dark theme, 3 panels) ─────────────────────
DARK_BG = "#0d1117"; PANEL_BG = "#161b22"; ALT_BG = "#1c2333"
GRID_COL = "#30363d"; TEXT_COL = "#e6edf3"; DIM_COL = "#8b949e"
RED = "#ff4d4f"; GREEN = "#3fb950"; GOLD = "#d4a017"; CYAN = "#58a6ff"

ret_pct = returns_clean * 100
RET_CAP = 1.5

fig_a, axes_a = plt.subplots(3, 1, figsize=(16, 10), sharex=True,
    gridspec_kw={"height_ratios": [1.5, 1.5, 1.5], "hspace": 0.06})
fig_a.patch.set_facecolor(DARK_BG)

for ax in axes_a:
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT_COL, labelsize=8)
    ax.grid(True, color=GRID_COL, linewidth=0.4, linestyle="--", alpha=0.5)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_COL)

for i, sess in enumerate(sessions):
    bg = ALT_BG if i % 2 == 1 else PANEL_BG
    for ax in axes_a:
        ax.axvspan(sess["x_start"], sess["x_end"] + 1, color=bg, alpha=1.0, zorder=0)

if oct19_sess:
    s19 = oct19_sess[0]
    for ax in axes_a:
        ax.axvspan(s19["x_start"], s19["x_end"] + 1, color=RED, alpha=0.12, zorder=1)

APPENDIX_HALT_BG = "#ffb3b3"
for halt_start, halt_end in halt_periods:
    if halt_start in df.index or halt_end in df.index:
        for ax in axes_a:
            ax.axvspan(dt_to_x(halt_start), dt_to_x(halt_end) + 1,
                       color=APPENDIX_HALT_BG, alpha=0.6,
                       linewidth=1, edgecolor=RED, linestyle="-", zorder=2)

for sess in sessions[1:]:
    for ax in axes_a:
        ax.axvline(sess["x_start"] - 0.5, color=DIM_COL, linewidth=1.2, linestyle="-", zorder=5)

# Panel 1 — returns
ax0 = axes_a[0]
pos_ret = np.clip(ret_pct.values,  0, RET_CAP)
neg_ret = np.clip(ret_pct.values, -RET_CAP, 0)
ax0.fill_between(x, pos_ret, color=CYAN, alpha=0.7, linewidth=0, zorder=3)
ax0.fill_between(x, neg_ret, color=RED,  alpha=0.7, linewidth=0, zorder=3)
ax0.axhline(0, color=GRID_COL, linewidth=0.8, zorder=4)
ax0.set_ylabel("1-min Return (%)", color=TEXT_COL, fontsize=9)
ax0.yaxis.label.set_color(TEXT_COL)
ax0.set_ylim(-RET_CAP, RET_CAP)
ax0.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.1f}%"))
ax0.text(5, RET_CAP * 0.82, "Intra-session only\n(overnight gaps removed)",
         color=DIM_COL, fontsize=7, va="top")
ax0.set_title("1987 Black Monday — Analytical Detail  (Oct 16–21, 1987)",
              color=TEXT_COL, fontsize=12, fontweight="bold", pad=10)

# Panel 2 — drawdown
ax1 = axes_a[1]
dd_pct = drawdown.values * 100
ax1.fill_between(x, dd_pct, 0, color=RED, alpha=0.45, linewidth=0, zorder=3)
ax1.plot(x, dd_pct, color=RED, linewidth=0.8, zorder=4)
ax1.set_ylabel("Drawdown (%)", color=TEXT_COL, fontsize=9)
ax1.yaxis.label.set_color(TEXT_COL)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax1.annotate(f"Max DD  {drawdown.min():.1%}",
             xy=(max_dd_x, drawdown.min() * 100),
             xytext=(max_dd_x - 80, drawdown.min() * 100 + 4),
             arrowprops=dict(arrowstyle="->", color=TEXT_COL, lw=1.0),
             color=TEXT_COL, fontsize=8, zorder=7)

# Panel 3 — rolling vol
ax2 = axes_a[2]
ax2.plot(x, rolling_vol.values, color=GOLD, linewidth=0.9, label="30-min rolling vol", zorder=3)
ax2.axhline(baseline_vol, color=GRID_COL, linewidth=0.8, linestyle="--",
            label=f"Pre-crash baseline ({baseline_vol:.4f})")
ax2.axhline(threshold, color=RED, linewidth=0.8, linestyle="--",
            label=f"3× threshold ({threshold:.4f})")
ax2.set_ylabel("Rolling Vol (std)", color=TEXT_COL, fontsize=9)
ax2.yaxis.label.set_color(TEXT_COL)
ax2.legend(loc="upper left", fontsize=7, facecolor=PANEL_BG,
           labelcolor=TEXT_COL, edgecolor=GRID_COL)

ax2.set_xticks([s["x_mid"] for s in sessions])
ax2.set_xticklabels([s["label"] for s in sessions], color=TEXT_COL, fontsize=9)
ax2.set_xlim(-5, len(df) + 5)

fig_a.text(0.5, 0.005,
           "Source: 1987_crash_market_data.csv  |  S&P 500 Futures, trading hours only (09:30–16:00)  |  Returns: intra-session only",
           ha="center", color=DIM_COL, fontsize=7)

plt.savefig(APPENDIX_PNG, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print(f"Figure 2 saved → {APPENDIX_PNG}")


## 4. Key Statistics

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────
rolling_60  = returns.rolling(60).sum()
worst_60_end   = rolling_60.idxmin()
worst_60_start = worst_60_end - pd.Timedelta(minutes=60)
vol_ratio      = rolling_vol.max() / baseline_vol

oct16_close = df.loc["1987-10-16"]["SP500_Futures"].dropna().iloc[-1]
oct19_close = df.loc["1987-10-19"]["SP500_Futures"].dropna().iloc[-1]
oct20_close = df.loc["1987-10-20"]["SP500_Futures"].dropna().iloc[-1]
oct21_close = df.loc["1987-10-21"]["SP500_Futures"].dropna().iloc[-1]

print("=" * 58)
print("1987 BLACK MONDAY — KEY STATISTICS")
print("=" * 58)
print(f"Oct 16 close (SP500 futures) : {oct16_close:.2f}")
print(f"Oct 19 close (SP500 futures) : {oct19_close:.2f}  ({(oct19_close-oct16_close)/oct16_close:+.1%} vs Fri)")
print(f"Oct 20 close (SP500 futures) : {oct20_close:.2f}  ({(oct20_close-oct19_close)/oct19_close:+.1%} vs Mon)")
print(f"Oct 21 close (SP500 futures) : {oct21_close:.2f}  ({(oct21_close-oct20_close)/oct20_close:+.1%} vs Tue)")
print()
print(f"Market break point           : {break_point}")
print(f"Peak-to-trough drawdown      : {drawdown.min():.2%}  at {drawdown.idxmin()}")
print(f"Worst 60-min window          : {rolling_60.min():.2%}  ({worst_60_start.strftime('%H:%M')} → {worst_60_end.strftime('%H:%M %b %d')})")
print(f"Peak 30-min rolling vol      : {rolling_vol.max():.4f}  at {rolling_vol.idxmax()}")
print(f"Pre-crash vol baseline       : {baseline_vol:.4f}")
print(f"Vol spike ratio              : {vol_ratio:.1f}× pre-crash baseline")
print(f"3× threshold first breached  : {break_by_vol}")
print()
print("Cross-asset moves (open-to-close, Oct 19):")
cross_asset = {
    "S&P 500 Futures": (oct19_close - oct16_close) / oct16_close,
    "DOW Futures (vs Oct 16)": None,
    "Treasuries": "+2.9% (flight to safety)",
    "Gold":       "+2.6% (flight to safety)",
    "Oil":        "–0.2% (no demand shock)",
    "Nikkei":     "–14.7% (lagged — crashed Oct 20 Asia time)",
}
for k, v in cross_asset.items():
    if isinstance(v, float):
        print(f"  {k:<30} {v:+.1%}")
    else:
        print(f"  {k:<30} {v}")


## 5. Memo — Full Text

---

### Executive Summary

Portfolio insurance was designed to protect institutional portfolios by reducing equity
exposure as markets fell. On October 19, 1987, that portfolio-level discipline became a
market-level vulnerability. As prices declined, many institutions received the same
mechanical sell signal simultaneously and attempted to execute into rapidly thinning
liquidity. What was rational for one portfolio became destabilising in the aggregate.

The failure was not a coding error. It was structural: a procyclical strategy deployed
at scale in a market that could no longer absorb the required trades.

---

### What the Data Show

The minute-level S&P 500 futures data indicate the market did not experience a normal
correction — it moved into disorderly liquidation. By 10:00 on October 19, rolling
volatility had already crossed **three times its pre-crash baseline**. The worst 60-minute
window (14:14–15:14) produced a cumulative loss of **11.2%**. By 13:01 on October 20,
cumulative drawdown from the pre-crash peak reached **28.6%**.

These figures are more consistent with disorderly liquidation than with a correction.

> **Figure 1** (above): S&P 500 Futures with PELT structural break at 09:42 on October 19.
> The sharp discontinuity signals the threshold beyond which selling pressure became irreversible.

---

### How Portfolio Insurance Worked

Portfolio insurance was designed to replicate downside protection without requiring investors
to buy a traditional put option. Instead of paying an upfront premium, institutions used
dynamic hedging to reduce market exposure as prices fell — most often through sales of S&P 500
futures (Presidential Task Force on Market Mechanisms, 1988).

The strategy assumed that large sell orders could be executed at or near quoted prices without
materially moving the market — that liquidity would hold at scale. On October 19, that
assumption failed almost immediately.

---

### Why the Algorithms Sold

The algorithms sold because that is what they were built to do. They were not designed to
identify undervaluation — they were designed to reduce exposure as prices fell.
**The models did not malfunction. They followed their rule set exactly as intended.**

The issue was scale and synchronisation. By late 1987, portfolio insurance strategies covered
an estimated **$60–90 billion** in assets (Carlson, 2007). Once the market declined sharply,
a large share of that capital faced the same imperative: sell futures to hedge additional
downside. The result was concentrated, largely price-insensitive execution into a market
already under pressure.

CFTC data underscore the magnitude: gross futures selling by institutional hedgers increased
from roughly **3,500 contracts** on October 14 to **32,700** on October 19 (CFTC, 1988).

> *"Falling prices triggered mechanical selling; that selling pushed prices lower; lower prices generated still more selling."* — Shiller (1988)

> *"Even if one investor could exit successfully, the system as a whole cannot all get out at once."* — Greenspan (1988)

---

### Risk-Control Architecture

**The Problem:** Portfolio insurance fails catastrophically not because it identifies too much
risk, but because it forces mechanical selling into collapsing liquidity — turning a circuit
breaker into a system amplifier. The pattern repeated in 2018's "Volmageddon."

**Proposed Solution: A Liquidity-Aware, Three-Layer Architecture**

| Layer | Mechanism | Mode |
|---|---|---|
| **Layer 1** | Volatility-Scaled Position Sizing | Continuous — leverage falls as vol rises |
| **Layer 2** | Liquidity-Aware Execution Limits | Conditional — volume caps, spread pause rules |
| **Layer 3** | Volatility-Conditioned Exit Thresholds | Backstop only — widens in stress, tightens in calm |

**Layer 1 — Volatility-Scaled Position Sizing (Continuous)**
Primary control. As volatility rises, position leverage automatically declines — *not* as a
binary trigger but continuously. The portfolio gradually shrinks its footprint as variance expands.

**Layer 2 — Liquidity-Aware Execution Limits (Conditional)**
Operational safeguard. Trade sizes capped at ≤2% of ADV. Automatic pause triggered by extreme
bid-ask spreads. Prevents mechanical crowding during stress.

**Layer 3 — Volatility-Conditioned Exit Thresholds (Backstop Only)**
Final circuit breaker — engages *only after* Layers 1 and 2 have already scaled down base exposure.
- Thresholds **widen** during high volatility (e.g., 8% loss level if VIX > 40)
- Thresholds **tighten** in calm markets (e.g., 3% loss level if VIX < 15)

**What This Solves:**
1. **Prevents liquidation cascades** — gradual scaling avoids the 1987/2018 syndrome
2. **Respects market capacity** — execution limits prevent algorithmic crowding
3. **Breaks the feedback loop** — entering stress already lighter means layers 2 and 3 protect what remains
4. **Maintains control hierarchy** — stop-loss rules are defensive backstops, not primary policy

**Key Trade-Offs:**
- *Whipsaw losses* in volatile-but-stable regimes (deliberate cost of tail protection)
- *Execution quality degradation* when pause rules engage (de-risking slower than ideal)
- *Overnight gap risk* — no dynamic framework prevents limit-down gaps
- *Calibration risk* — thresholds must be market-specific and regime-dependent

**Bottom Line:**

> The fundamental shift is from *"act when price falls to X"* to *"continuously scale inversely
> to stress, execute respectfully, and only exit if all else fails."* No framework eliminates all
> losses, but this architecture materially reduces the probability of forced liquidation during
> the exact moments when market capacity is most depleted.

---


## 6. References

- Augustin, P., Cheng, I.-H., & Van den Bergen, L. (2021). Volmageddon and the failure of short volatility products. *Financial Analysts Journal, 77*(3), 35–51.
- Carlson, M. A. (2007). *A brief history of the 1987 stock market crash with a discussion of the Federal Reserve response.* Finance and Economics Discussion Series 2007-13. Board of Governors of the Federal Reserve System.
- Commodity Futures Trading Commission. (1988, January). *Final report on stock index futures and cash market activity during October 1987.* Washington, DC: CFTC.
- Fortune, P. (1993). Stock market crashes: What have we learned from October 1987? *New England Economic Review*, March/April, 13–24.
- Greenspan, A. (1988, December 28). *Remarks before a joint meeting of the American Economic Association and the American Finance Association.* New York, NY.
- Haldane, A. G., & May, R. M. (2011). Systemic risk in banking ecosystems. *Nature, 469*(7330), 351–355.
- Moreira, A., & Muir, T. (2017). Volatility-managed portfolios. *The Journal of Finance, 72*(4), 1611–1644.
- Presidential Task Force on Market Mechanisms. (1988, January). *Report of the Presidential Task Force on Market Mechanisms.* U.S. Department of the Treasury.
- Shiller, R. J. (1988). Portfolio insurance and other investor fashions as factors in the 1987 stock market crash. In S. Fischer (Ed.), *NBER Macroeconomics Annual 1988, Volume 3* (pp. 287–297). MIT Press.
